# تدريب ALLaM-7B-Instruct-preview على داتا مداد — QLoRA (Kaggle)

هذا النوتبوك يدرّب نموذج **ALLaM-7B-Instruct-preview** (SDAIA/HUMAIN، رخصة
Apache 2.0) على داتا القصص التعليمية المُدمجة (١٧٧ مثال، ٦ مواد، ٣٧ درساً)
باستخدام **QLoRA** (4-bit quantization + LoRA adapters) — هذا الأسلوب الأنسب
لدرجم بحجم صغير نسبياً لأنه:

- يدرّب عدد قليل جداً من المعاملات (LoRA rank منخفض) → يقلل خطر الحفظ الحرفي (overfitting).
- يوقف التدريب تلقائياً (early stopping) أول ما يبدأ الأداء على مجموعة التحقق يسوء.
- يراقب `eval_loss` كل epoch ويحتفظ بأفضل نسخة فقط.

## قبل التشغيل على Kaggle
1. **الإعدادات (Settings) → Accelerator**: اختاري `GPU T4 x2` (أو أي GPU متاح).
2. **Settings → Internet**: شغّليه (On) — محتاجينه لتحميل النموذج من Hugging Face.
3. ارفعي ملفات `merged_train.jsonl` و`merged_validation.jsonl` و`merged_test.jsonl`
   كـ **Kaggle Dataset** جديد (New Dataset → ارفعي الملفات الثلاثة)، وعدّلي
   `DATA_DIR` بالخلية تحت حسب اسم الداتاسِت اللي طلع لك (يبان بـ `/kaggle/input/<اسم-الداتاسِت>`).
4. لو نموذج ALLaM يحتاج تسجيل دخول (gated) على Hugging Face، أضيفي التوكن
   من **Add-ons → Secrets** باسم `HF_TOKEN`.


## ١. تثبيت المكتبات

In [ ]:
!pip install -q -U "transformers>=4.43.0" "peft>=0.11.1" "trl>=0.9.6" \
    "bitsandbytes>=0.43.1" "accelerate>=0.32.0" "datasets>=2.20.0" sentencepiece


In [ ]:
import transformers
print(transformers.__version__)

## ٢. الإعدادات العامة

In [ ]:
import os, json, random
import numpy as np
import torch


DATA_DIR = "/kaggle/input/datasets/lama377/midad-training-data" 

TRAIN_PATH = f"{DATA_DIR}/merged_train.jsonl"
VAL_PATH   = f"{DATA_DIR}/merged_validation.jsonl"
TEST_PATH  = f"{DATA_DIR}/merged_test.jsonl"

MODEL_ID = "humain-ai/ALLaM-7B-Instruct-preview"  
OUTPUT_DIR = "/kaggle/working/midad-allam-lora"
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("ما لقيت HF_TOKEN بالـ Secrets — تجاهلي هذا لو النموذج مو gated.")

print("GPU متاح:", torch.cuda.is_available(), "| عدد الـ GPUs:", torch.cuda.device_count())


## ٣. تحميل الداتا

In [ ]:
from datasets import Dataset

def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = read_jsonl(TRAIN_PATH)
val_rows   = read_jsonl(VAL_PATH)
test_rows  = read_jsonl(TEST_PATH)

print(f"train={len(train_rows)}  validation={len(val_rows)}  test={len(test_rows)}")

train_ds = Dataset.from_list(train_rows)
val_ds   = Dataset.from_list(val_rows)
test_ds  = Dataset.from_list(test_rows)


## ٤. تحميل التوكنايزر والنموذج (4-bit QLoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True,
)
model.config.use_cache = False
model.config.pretraining_tp = 1


## ٥. تنسيق الأمثلة بصيغة الحوار (chat template)



In [ ]:
def format_example(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_ds_fmt = train_ds.map(format_example, remove_columns=train_ds.column_names)
val_ds_fmt   = val_ds.map(format_example, remove_columns=val_ds.column_names)

print(train_ds_fmt[0]["text"][:800])


## ٦. فحص أطوال الأمثلة (لاختيار `max_seq_length` بدقة)

بعض القصص طويلة (مثل قصص تطبيق مداد الجاهزة)، فنحسب أطول عدد توكِن فعلي
بدل ما نخمّن رقم عشوائي.

In [ ]:
lengths = [len(tokenizer(t)["input_ids"]) for t in train_ds_fmt["text"]]
print("أقصى طول:", max(lengths), "| متوسط:", int(sum(lengths)/len(lengths)), "| أطول ١٠٪:", int(np.percentile(lengths, 90)))

MAX_SEQ_LENGTH = int(np.percentile(lengths, 99)) + 64 
MAX_SEQ_LENGTH = max(512, min(MAX_SEQ_LENGTH, 4096))
print("MAX_SEQ_LENGTH المُختار:", MAX_SEQ_LENGTH)


## ٧. إعداد LoRA — بإعدادات محافِظة تقلل خطر الـ overfitting

- `r=16` (منخفض نسبياً — كل ما زاد رفع احتمال الحفظ الحرفي مع داتا صغيرة).
- `lora_dropout=0.05` يمنع الاعتماد الزائد على أنماط معينة بالداتا.
- نستهدف طبقات الـ attention والـ MLP الأساسية فقط (النمط المعتاد لنماذج Llama-style).

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## ٨. إعدادات التدريب — Early Stopping على `eval_loss`

- `num_train_epochs=3` كحد أقصى، لكن `EarlyStoppingCallback` يوقف التدريب
  فور ما `eval_loss` يتوقف عن التحسّن لجولة تقييم وحدة (`patience=1`) — يعني
  ما يكمل يحفظ حرفياً بعد ما يبدأ يسوء على بيانات ما شافها.
- `load_best_model_at_end=True` يضمن إن آخر نسخة محفوظة فعلياً هي الأفضل على
  validation، مو آخر epoch بالضرورة.
- `learning_rate` محافظ (١.٥×10⁻⁴) بدل قيم أعلى شائعة، لأن الداتا صغيرة.

In [ ]:
from trl import SFTConfig
from transformers import EarlyStoppingCallback

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1.5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=SEED,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    loss_type="nll",
)

early_stopping = EarlyStoppingCallback(early_stopping_patience=1)

## ٩. بناء الـ Trainer وبدء التدريب

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_fmt,
    eval_dataset=val_ds_fmt,
    processing_class=tokenizer,
    callbacks=[early_stopping],
)

trainer.train()

## ١٠. رسم منحنى الخسارة (تأكيد بصري إنه ما صار overfitting)

لو `eval_loss` بدأ يرتفع بعد ما كان ينزل مع `train_loss`، هذا دليل overfitting
واضح — لو صار كذا، إما قلّلي الـ epochs أو خففي rank الـ LoRA أكثر (مثلاً ٨).

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_pts = [(h["epoch"], h["loss"]) for h in history if "loss" in h]
eval_pts  = [(h["epoch"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(7, 4))
if train_pts:
    plt.plot(*zip(*train_pts), label="train_loss", marker="o", markersize=3)
if eval_pts:
    plt.plot(*zip(*eval_pts), label="eval_loss", marker="s")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("منحنى الخسارة — تدريب مقابل تحقق")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png", dpi=120, bbox_inches="tight")
plt.show()


## ١١. حفظ الـ LoRA adapter

In [ ]:
FINAL_DIR = f"{OUTPUT_DIR}/final_adapter"
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print("تم الحفظ في:", FINAL_DIR)


## ١٢. اختبار توليد على درس "مفاجئ" غير موجود بالداتا

هذا أهم فحص عملي للـ overfitting: نجرب درساً ما شافه النموذج إطلاقاً أثناء
التدريب، ونشوف هل يرجع قصة جديدة متماسكة بنفس الأسلوب (تعميم صحي) ولا يرجّع
نص شبه حرفي من قصة تدريب ثانية (علامة حفظ).

In [ ]:
model.eval()

new_lesson_prompt = (
    "عنوان الدرس:\n"
    "أنواع الطاقة المتجددة\n\n"
    "أهداف التعلم:\n"
    "* أن يذكر الطفل مصدرين من مصادر الطاقة المتجددة (الشمس والرياح).\n"
    "* أن يوضح الطفل فائدة الطاقة المتجددة للبيئة.\n\n"
    "المفاهيم الأساسية:\n"
    "* الطاقة الشمسية\n"
    "* طاقة الرياح\n"
    "* الطاقة المتجددة\n\n"
    "المحتوى التعليمي:\n"
    "الطاقة المتجددة هي طاقة نحصل عليها من مصادر طبيعية لا تنفد، مثل الشمس والرياح، "
    "وهي صديقة للبيئة لأنها لا تسبب تلوثاً.\n\n"
    "المهمة:\n"
    "حوّل هذا الدرس إلى قصة تعليمية ممتعة ومناسبة لطفل سعودي عمره من 7 إلى 10 سنوات، "
    "مع المحافظة على دقة المعلومات التعليمية."
)

messages = [
    {"role": "system", "content": "أنت كاتب قصص تعليمية للأطفال من عمر 7 إلى 10 سنوات. مهمتك تحويل الدروس التعليمية إلى قصص ممتعة وسهلة الفهم مع المحافظة على دقة المعلومات وإضافة لمسات طبيعية من البيئة والثقافة السعودية. عند ذكر مكان القصة، استخدمي دائماً اسم مدينة أو منطقة سعودية حقيقية ومميزة."},
    {"role": "user", "content": new_lesson_prompt},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.pad_token_id,
    )

generated = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(generated)

In [ ]:
!pip install -q -U torchao 

## ١٣. (اختياري) دمج LoRA بالنموذج الأساسي وحفظ نسخة كاملة

مفيد لو تبين تصدّرين النموذج جاهز بدون الحاجة لـ `peft` وقت الاستدلال (inference)،
مثلاً لرفعه أو تشغيله بسرعة أكبر وقت العرض بالهاكاثون.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_ID = "humain-ai/ALLaM-7B-Instruct-preview"
OUTPUT_DIR = "/kaggle/working/midad-allam-lora"
FINAL_DIR = f"{OUTPUT_DIR}/final_adapter"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model_bf16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

merged_model = PeftModel.from_pretrained(
    base_model_bf16,
    FINAL_DIR
)

merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained(
    f"{OUTPUT_DIR}/merged_full_model"
)

tokenizer.save_pretrained(
    f"{OUTPUT_DIR}/merged_full_model"
)

print("تم دمج النموذج الجديد وحفظه.")